In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path


In [65]:
# conectar con la base de datos y cargar las 3 tablas
import sqlite3

conn = sqlite3.connect('../datos/intermedios/mercado_inmobiliario.db')

listings = pd.read_sql_query("SELECT * FROM listings", conn)
listings_det = pd.read_sql_query("SELECT * FROM listings_det", conn)
precio = pd.read_sql_query("SELECT * FROM idealista", conn)

conn.close()

In [66]:
# Merge inner join entre listings y listings_det
df_listings = pd.merge(listings, listings_det, on='id', how='inner')

In [67]:
# Merge entre df_listings y precio (left join) usando las claves correctas
df = pd.merge(df_listings, precio, left_on='neighbourhood_group', right_on='distrito', how='left')

In [68]:
# eliminar neighbourhood_group de df
df = df.drop(columns=['id','neighbourhood_group'])

In [69]:
# renombrar price a precio_noche y precio a precio_m2
df = df.rename(columns={'price': 'precio_noche', 'precio': 'precio_m2'})

In [76]:
# eliminar filas con availability_365 menor que 180
df = df[df['availability_365'] >= 180]

In [78]:
df.head()


,name,neighbourhood,latitude,longitude,room_type,precio_noche,minimum_nights,availability_365,description,accommodates,bathrooms,bedrooms,beds,review_scores_location,estimated_occupancy_l365d,precio_m2,distrito
0,MAGIC ARTISTIC HOUSE IN THE CENTER OF MADRID,Justicia,40.41884,-3.69655,Private room,64.0,1,314,INCREDIBLE HOME OF AN ARTIST SURROUNDED BY PAI...,4,1.5,1.0,2.0,4.97,204,7281,Centro
6,ROOM IN THE CENTER -LA LATINA- WIFI,Palacio,40.41143,-3.70912,Private room,47.0,1,259,None,2,1.0,1.0,2.0,4.95,222,7281,Centro
7,private house B & B. Arturo Soria (Metro),Piovera,40.45575,-3.64912,Private room,90.0,1,331,"Residential area, very quite house, no more gu...",2,1.0,1.0,2.0,4.84,54,5049,Hortaleza
12,Tafari Gran Via,Universidad,40.42116,-3.70384,Entire home/apt,106.0,15,269,None,2,1.0,0.0,1.0,4.94,255,7281,Centro
13,Private Room in a Sunny and Luxuriously Appoin...,Canillas,40.45959,-3.64699,Private room,68.0,2,275,Set a traditional dining table and enjoy a del...,2,1.0,1.0,1.0,4.82,162,5049,Hortaleza


In [79]:
tasa_uso = 0.5

condiciones = [
    df['room_type'] == 'Entire home/apt',
    (df['room_type'] == 'Private room') & (df['bedrooms'] == 1),
    (df['room_type'] == 'Private room') & (df['bedrooms'] > 1),
    (df['room_type'] == 'Shared room') & (df['accommodates'] == 1),
    (df['room_type'] == 'Shared room') & (df['accommodates'] > 1)
]
opciones = [
    df['precio_noche'],
    df['precio_noche'] * df['bedrooms'],
    df['precio_noche'] * df['bedrooms'] * tasa_uso,
    df['precio_noche'] * df['accommodates'],
    df['precio_noche'] * df['accommodates'] * tasa_uso
]
df['precio_noche_total'] = np.select(condiciones, opciones, default=np.nan)

In [80]:
# crear una variable ingreso_anual como la multiplicacion de precio_noche_total * estimated_occupancy_l365d
df['ingreso_anual'] = df['precio_noche_total'] * df['estimated_occupancy_l365d']

In [81]:
df.head()

,name,neighbourhood,latitude,longitude,room_type,precio_noche,minimum_nights,availability_365,description,accommodates,bathrooms,bedrooms,beds,review_scores_location,estimated_occupancy_l365d,precio_m2,distrito,precio_noche_total,ingreso_anual
0,MAGIC ARTISTIC HOUSE IN THE CENTER OF MADRID,Justicia,40.41884,-3.69655,Private room,64.0,1,314,INCREDIBLE HOME OF AN ARTIST SURROUNDED BY PAI...,4,1.5,1.0,2.0,4.97,204,7281,Centro,64.0,13056.0
6,ROOM IN THE CENTER -LA LATINA- WIFI,Palacio,40.41143,-3.70912,Private room,47.0,1,259,None,2,1.0,1.0,2.0,4.95,222,7281,Centro,47.0,10434.0
7,private house B & B. Arturo Soria (Metro),Piovera,40.45575,-3.64912,Private room,90.0,1,331,"Residential area, very quite house, no more gu...",2,1.0,1.0,2.0,4.84,54,5049,Hortaleza,90.0,4860.0
12,Tafari Gran Via,Universidad,40.42116,-3.70384,Entire home/apt,106.0,15,269,None,2,1.0,0.0,1.0,4.94,255,7281,Centro,106.0,27030.0
13,Private Room in a Sunny and Luxuriously Appoin...,Canillas,40.45959,-3.64699,Private room,68.0,2,275,Set a traditional dining table and enjoy a del...,2,1.0,1.0,1.0,4.82,162,5049,Hortaleza,68.0,11016.0


In [82]:
# Crear variable m2_estimado según reglas de negocio usando np.select
condiciones_m2 = [
    (df['bedrooms'] <= 1),
    (df['bedrooms'] == 2),
    (df['bedrooms'] == 3) & (df['bathrooms'] < 2),
    (df['bedrooms'] == 3) & (df['bathrooms'] >= 2),
    (df['bedrooms'] > 3) | (df['bathrooms'] >= 3)
]
opciones_m2 = [
    50,
    65,
    90,
    110,
    140
]
df['m2_estimado'] = np.select(condiciones_m2, opciones_m2, default=np.nan)

In [83]:
# Crear una nueva variable llamada coste_adquisicion que sea el resultado de multiplicar m2_estimado * precio_m2 y multiplicar por un ajuste de 0.75
df['coste_adquisicion'] = df['m2_estimado'] * df['precio_m2'] * 0.75    

In [84]:
# Calcular atractivo turístico para cada registro
import sys
sys.path.append('..')
from funciones.funciones import atractivo_turistico_0_100

In [85]:
poi_df = pd.read_csv('../datos/brutos/poi_madrid.csv')
df['atractivo_turistico'] = df.apply(lambda row: atractivo_turistico_0_100(row['latitude'], row['longitude'], poi_df), axis=1)

In [86]:
# discretiza bedrooms en las categorias '0', '1', '2', '3', '4+'
df['bedrooms_disc'] = pd.cut(df['bedrooms'], bins=[-1,0, 1, 2, 3, np.inf], labels=['0_hab', '1_hab', '2_hab', '3_hab', '4+_hab'])

In [87]:
# discretiza bathrooms en las categorías '<=1', '1-2', '>2'
df['bathrooms_disc'] = pd.cut(df['bathrooms'], bins=[-np.inf, 1, 2, np.inf], labels=['01_Uno', '02_Dos', '03_Mas_de_dos'])

In [88]:
# discretiza accommodates en las categorias '1', '2', '3', '4','5+'
df['accommodates_disc'] = pd.cut(df['accommodates'], bins=[0, 1, 2, 3, 4, np.inf], labels=['01_Uno', '02_Dos', '03_Tres', '04_Cuatro', '05_Mas_de_Cuatro'])

In [89]:
# discretiza beds en las categorías '<=1','2','3','4','>4'
df['beds_disc'] = pd.cut(df['beds'], bins=[-1, 1, 2, 3, 4, 100], labels=['1 cama', '2 camas', '3 camas', '4 camas', '5_o_mas_camas'])


In [90]:
# crea una variable margen_bruto que sea el resultado de dividir ingreso_anual entre coste_adquisicion
df['margen_bruto'] = df['ingreso_anual'] / df['coste_adquisicion'] * 100  # expresado en porcentaje y redondeado a 2 decimales
df['margen_bruto'] = df['margen_bruto'].round(2)

In [91]:
df.head()

,name,neighbourhood,latitude,longitude,room_type,precio_noche,minimum_nights,availability_365,description,accommodates,...,precio_noche_total,ingreso_anual,m2_estimado,coste_adquisicion,atractivo_turistico,bedrooms_disc,bathrooms_disc,accommodates_disc,beds_disc,margen_bruto
0,MAGIC ARTISTIC HOUSE IN THE CENTER OF MADRID,Justicia,40.41884,-3.69655,Private room,64.0,1,314,INCREDIBLE HOME OF AN ARTIST SURROUNDED BY PAI...,4,...,64.0,13056.0,50.0,273037.5,14.987660,1_hab,02_Dos,04_Cuatro,2 camas,4.78
6,ROOM IN THE CENTER -LA LATINA- WIFI,Palacio,40.41143,-3.70912,Private room,47.0,1,259,None,2,...,47.0,10434.0,50.0,273037.5,10.305154,1_hab,01_Uno,02_Dos,2 camas,3.82
7,private house B & B. Arturo Soria (Metro),Piovera,40.45575,-3.64912,Private room,90.0,1,331,"Residential area, very quite house, no more gu...",2,...,90.0,4860.0,50.0,189337.5,0.000000,1_hab,01_Uno,02_Dos,2 camas,2.57
12,Tafari Gran Via,Universidad,40.42116,-3.70384,Entire home/apt,106.0,15,269,None,2,...,106.0,27030.0,50.0,273037.5,15.005323,0_hab,01_Uno,02_Dos,1 cama,9.90
13,Private Room in a Sunny and Luxuriously Appoin...,Canillas,40.45959,-3.64699,Private room,68.0,2,275,Set a traditional dining table and enjoy a del...,2,...,68.0,11016.0,50.0,189337.5,0.000000,1_hab,01_Uno,02_Dos,1 cama,5.82


In [95]:
# crear una nueva tabla en la base de datos mercado_inmobiliario.db que se llame tablon_analitico con el dataframe df
conn = sqlite3.connect('../datos/intermedios/mercado_inmobiliario.db')
df.to_sql('tablon_analitico', conn, if_exists='replace', index=False)
conn.close()

## Exportación del tablón analítico a CSV

Se exporta el tablón analítico al formato CSV para su uso en la aplicación Streamlit de visualización.

Antes de exportar se aplica una limpieza de campos de texto (`name`, `description`) para evitar problemas de alineación de columnas en herramientas externas (Excel, etc.) causados por comas y comillas internas no escapadas:

- **`name`**: comas → ` · ` ; comillas dobles → comillas simples ; saltos de línea → espacio
- **`description`**: eliminación de etiquetas HTML (`<br />`, `<b>`, etc.) ; mismo tratamiento que `name`

El CSV resultante mantiene exactamente las **27 columnas** del tablón y es compatible con `pd.read_csv()` sin parámetros adicionales.

In [ ]:
import re
from pathlib import Path

# ── Copia de trabajo para no alterar df original ─────────────────────────────
df_csv = df.copy()

# ── Limpieza del campo 'name' ─────────────────────────────────────────────────
def clean_name(s):
    """Elimina caracteres problemáticos para CSV: comas, comillas dobles y saltos de línea."""
    s = str(s) if s is not None else ''
    s = s.replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ')  # saltos de línea
    s = s.replace('"', "'")                                            # comillas dobles → simples
    s = s.replace(',', ' ·')                                           # comas → punto medio
    s = re.sub(r'\s{2,}', ' ', s)                                      # espacios múltiples → uno
    return s.strip()

df_csv['name'] = df_csv['name'].apply(clean_name)

# ── Limpieza del campo 'description' ─────────────────────────────────────────
def clean_description(s):
    """Elimina etiquetas HTML y aplica la misma limpieza que clean_name."""
    s = str(s) if s is not None else ''
    s = re.sub(r'<[^>]+>', ' ', s)   # quitar tags HTML (<br />, <b>, <a ...>, etc.)
    return clean_name(s)             # reutilizar limpieza de comas, comillas y saltos

df_csv['description'] = df_csv['description'].apply(clean_description)

# ── Selección y orden de columnas (27 columnas del tablón analítico) ──────────
columnas_tablon = [
    'name', 'neighbourhood', 'latitude', 'longitude', 'room_type',
    'precio_noche', 'minimum_nights', 'availability_365', 'description',
    'accommodates', 'bathrooms', 'bedrooms', 'beds', 'review_scores_location',
    'estimated_occupancy_l365d', 'precio_m2', 'distrito',
    'precio_noche_total', 'ingreso_anual', 'm2_estimado', 'coste_adquisicion',
    'atractivo_turistico', 'bedrooms_disc', 'bathrooms_disc',
    'accommodates_disc', 'beds_disc', 'margen_bruto'
]
df_csv = df_csv[columnas_tablon]

# ── Rutas de salida ───────────────────────────────────────────────────────────
notebook_dir = Path().resolve()                           # directorio del notebook
out_intermedios = notebook_dir / '../datos/intermedios/tablon_analitico.csv'
out_streamlit   = notebook_dir / '../../streamlit_app/data/tablon_analitico.csv'

# Guardar en datos/intermedios (salida estándar del pipeline)
out_intermedios.resolve().parent.mkdir(parents=True, exist_ok=True)
df_csv.to_csv(out_intermedios, index=False)
print(f"✅ CSV guardado en: {out_intermedios.resolve()}")
print(f"   Dimensiones: {df_csv.shape[0]:,} filas × {df_csv.shape[1]} columnas")

# Guardar también en la carpeta de la app Streamlit (si existe)
if out_streamlit.resolve().parent.exists():
    df_csv.to_csv(out_streamlit, index=False)
    print(f"✅ CSV copiado a la app Streamlit: {out_streamlit.resolve()}")
else:
    print(f"ℹ️  Carpeta de la app Streamlit no encontrada — solo se guardó en datos/intermedios")

# ── Verificación rápida ───────────────────────────────────────────────────────
df_check = pd.read_csv(out_intermedios)
n_extra_cols = len([c for c in df_check.columns if 'Unnamed' in str(c)])
print(f"\n📋 Verificación del CSV generado:")
print(f"   Columnas: {list(df_check.columns)}")
print(f"   Columnas extra (Unnamed): {n_extra_cols}")
print(f"   Filas con NaN en margen_bruto: {df_check['margen_bruto'].isna().sum()}")